- Install AutoML libraries
- Note: These have many dependencies, installation may take time

In [ ]:
# Install AutoML libraries
# Note: These have many dependencies, installation may take time
!pip install -q pycaret flaml optuna

**Random Seed:** Setting a seed ensures **reproducibility** — the same random numbers are generated each time the code runs. This is critical in ML experiments because:
- Train/test splits will be the same
- Weight initialization will be identical
- Any randomized algorithm (dropout, data augmentation) will behave consistently

Without a fixed seed, your results would vary between runs, making it impossible to debug or compare experiments.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed
np.random.seed(42)

## Part 1: Introduction to AutoML

### What is AutoML?

**Automated Machine Learning (AutoML)** automates the end-to-end process of applying ML to real-world problems.

**AutoML handles:**
- ✅ Data preprocessing
- ✅ Feature engineering
- ✅ Model selection
- ✅ Hyperparameter tuning
- ✅ Ensemble methods
- ✅ Model evaluation

**Benefits:**
- ⚡ Faster prototyping
- 🎯 Finds good baselines
- 🔍 Explores many models automatically
- 📊 Automates tedious tasks
- 🎓 Good for learning

**When to Use:**
- Quick baseline needed
- Exploring new datasets
- Limited ML expertise
- Time constraints
- Model comparison

**When NOT to Use:**
- Highly specialized problems
- Need full control
- Production-critical systems (without review)
- Very large datasets (computationally expensive)

## Part 2: PyCaret - Low-Code ML Library

PyCaret is a Python library that wraps several ML libraries for low-code experience.

In [ ]:
# Load classification dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
df.head()

Import the libraries needed for this section:
- **pycaret**
- *****

In [ ]:
from pycaret.classification import *

# Setup PyCaret environment
# This does preprocessing, train/test split, and more
clf_setup = setup(
    data=df,
    target='target',
    session_id=42,
    verbose=False,
    normalize=True,           # Normalize features
    remove_outliers=False,    # Keep outliers for now
    train_size=0.8           # 80-20 split
)

print("Setup complete!")

Compare all available models
This trains and evaluates 10+ models!

The code below implements this step in the low-code AI workflow.

In [ ]:
# Compare all available models
# This trains and evaluates 10+ models!
print("Comparing models... (this may take a minute)")
best_models = compare_models(n_select=5, sort='AUC')

print("\n✅ Top 5 models identified!")

Get the best model
Evaluate the model

The code below implements this step in the low-code AI workflow.

In [ ]:
# Get the best model
best_model = best_models[0]

print(f"Best model: {best_model}")

# Evaluate the model
print("\nModel evaluation:")
evaluate_model(best_model)

### Tune hyperparameters

Define variables and print their values to verify the results.

In [ ]:
# Tune hyperparameters
print("Tuning hyperparameters...")
tuned_model = tune_model(
    best_model,
    optimize='AUC',
    n_iter=10  # Number of iterations
)

print("\n✅ Model tuned!")

**Classification Report:** A summary of key metrics per class:

- **Precision**: Of all items predicted as class X, what fraction actually are? (Low precision = many false positives)
- **Recall**: Of all actual class X items, what fraction did we catch? (Low recall = many false negatives)
- **F1-Score**: Harmonic mean of precision and recall — balances both
- **Support**: Number of actual instances per class

In [ ]:
# Make predictions
holdout_pred = predict_model(tuned_model)

print("Predictions on holdout set:")
print(holdout_pred.head())

# Get metrics
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(
    holdout_pred['target'],
    holdout_pred['prediction_label']
))

### Create ensemble

Final predictions

In [ ]:
# Create ensemble
print("Creating ensemble...")
ensemble_model = ensemble_model(tuned_model, method='Bagging')

print("\n✅ Ensemble created!")

# Final predictions
final_pred = predict_model(ensemble_model)
print(f"\nFinal ensemble performance on holdout:")
print(f"Accuracy: {(final_pred['prediction_label'] == final_pred['target']).mean():.4f}")

### Save the model

Load it back (to demonstrate)

In [ ]:
# Save the model
save_model(ensemble_model, 'pycaret_best_model')
print("Model saved as 'pycaret_best_model.pkl'")

# Load it back (to demonstrate)
loaded_model = load_model('pycaret_best_model')
print("Model loaded successfully!")

## Part 3: FLAML - Fast Lightweight AutoML

FLAML is optimized for speed and resource efficiency.

In [ ]:
from flaml import AutoML

# Prepare data
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
# Initialize FLAML AutoML
automl = AutoML()

# Configure and run
settings = {
    "time_budget": 60,  # seconds
    "metric": 'roc_auc',
    "task": 'classification',
    "log_file_name": 'flaml_experiment.log',
    "seed": 42
}

print("Running FLAML AutoML (60 second budget)...")
automl.fit(X_train, y_train, **settings)

print("\n✅ AutoML complete!")

Print best model and configuration

The code below implements this step in the low-code AI workflow.

In [ ]:
# Print best model and configuration
print("Best model found:")
print(f"  Algorithm: {automl.best_estimator}")
print(f"  ROC-AUC: {automl.best_loss:.4f}")
print(f"\nBest hyperparameters:")
for param, value in automl.best_config.items():
    print(f"  {param}: {value}")

**Model Prediction (.predict()):** After training, `.predict()` applies the learned model to new (unseen) data to generate predictions.

- **Classification**: Returns predicted class labels
- **Regression**: Returns predicted continuous values

The quality of predictions depends on how well the model was trained and whether the new data resembles the training distribution.

**Accuracy:** The fraction of predictions that are correct:

$$\text{Accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}$$

**Caution:** Accuracy can be misleading with imbalanced classes. If 95% of data is class A, a model that always predicts 'A' gets 95% accuracy while being useless. Use precision, recall, and F1-score for imbalanced datasets.

In [ ]:
# Evaluate on test set
from sklearn.metrics import roc_auc_score, accuracy_score

y_pred = automl.predict(X_test)
y_pred_proba = automl.predict_proba(X_test)[:, 1]

print("Test set performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

**Create a DataFrame:** Pandas DataFrames are 2D labeled data structures — they're the backbone of data science in Python. Each column can hold a different data type (numbers, strings, dates). DataFrames support powerful operations: filtering, grouping, merging, reshaping, and aggregation.

In [ ]:
# Visualize feature importance
if hasattr(automl.model.estimator, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': automl.model.estimator.feature_importances_
    }).sort_values('importance', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'], feature_importance['importance'])
    plt.xlabel('Importance')
    plt.title('Top 10 Feature Importances (FLAML)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type")

## Part 4: Regression with AutoML

Let's try AutoML on a regression problem.

In [ ]:
# Load regression dataset
diabetes = load_diabetes()
df_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df_reg['target'] = diabetes.target

print(f"Dataset shape: {df_reg.shape}")
print(f"\nTarget statistics:")
print(df_reg['target'].describe())
df_reg.head()

### PyCaret for regression

Define variables and print their values to verify the results.

In [ ]:
# PyCaret for regression
from pycaret.regression import *

reg_setup = setup(
    data=df_reg,
    target='target',
    session_id=42,
    verbose=False,
    normalize=True,
    train_size=0.8
)

print("Regression setup complete!")

### Compare regression models

Define variables and print their values to verify the results.

In [ ]:
# Compare regression models
print("Comparing regression models...")
best_reg_models = compare_models(n_select=3, sort='R2')

print("\n✅ Top 3 regression models identified!")

### Tune the best model

Define variables and print their values to verify the results.

In [ ]:
# Tune the best model
best_reg = best_reg_models[0]
print(f"Tuning {best_reg}...")

tuned_reg = tune_model(best_reg, optimize='R2', n_iter=10)
print("\n✅ Model tuned!")

**Mean Squared Error (MSE) Loss:** The go-to loss for **regression** — predicting continuous values:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Squaring the errors means large mistakes are penalized much more than small ones. This makes MSE sensitive to outliers. For outlier-robust regression, consider MAE (Mean Absolute Error) or Huber loss.

**R² Score (Coefficient of Determination):** Measures how much of the variance in the target variable is explained by the model:

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- $R^2 = 1$: Perfect predictions
- $R^2 = 0$: Model is no better than predicting the mean
- $R^2 < 0$: Model is worse than predicting the mean

In [ ]:
# Evaluate regression model
holdout_reg = predict_model(tuned_reg)

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("Regression metrics on holdout:")
print(f"  R² Score: {r2_score(holdout_reg['target'], holdout_reg['prediction_label']):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(holdout_reg['target'], holdout_reg['prediction_label'])):.4f}")
print(f"  MAE: {mean_absolute_error(holdout_reg['target'], holdout_reg['prediction_label']):.4f}")

**Scatter Plot:** Displays individual data points as dots on a 2D plane. Each point's position is determined by its x and y values. Color and size can encode additional dimensions.

**Use for:** Exploring relationships between two continuous variables, spotting clusters, identifying outliers, and visualizing model predictions vs. actual values.

**Line Plot:** Connects data points with lines — ideal for showing trends over a continuous variable (often time). In ML, line plots are commonly used for:
- Training/validation loss curves (to detect overfitting)
- Learning rate schedules
- Time series data

In [ ]:
# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(
    holdout_reg['target'],
    holdout_reg['prediction_label'],
    alpha=0.6,
    edgecolors='k'
)
plt.plot(
    [holdout_reg['target'].min(), holdout_reg['target'].max()],
    [holdout_reg['target'].min(), holdout_reg['target'].max()],
    'r--',
    lw=2,
    label='Perfect Prediction'
)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted (Regression AutoML)')
plt.legend()
plt.tight_layout()
plt.show()

## Part 5: Comparing AutoML Platforms

Let's compare different AutoML approaches on the same dataset.

In [ ]:
import time

# Prepare data
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

results = []

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
# 1. FLAML
print("Testing FLAML...")
start = time.time()

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task='classification',
    metric='roc_auc',
    time_budget=30,
    verbose=0
)

flaml_time = time.time() - start
flaml_pred = flaml_automl.predict_proba(X_test)[:, 1]
flaml_score = roc_auc_score(y_test, flaml_pred)

results.append({
    'Platform': 'FLAML',
    'Time (s)': flaml_time,
    'ROC-AUC': flaml_score,
    'Best Model': flaml_automl.best_estimator
})

print(f"✅ FLAML: {flaml_score:.4f} in {flaml_time:.2f}s")

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

**Fit and Transform (.fit_transform()):** A convenience method that combines `.fit()` and `.transform()` in one step — learns the transformation parameters from the data and immediately applies the transformation.

**Important**: Use `.fit_transform()` only on **training data**. For test data, use `.transform()` alone to apply the same transformation learned from training. Otherwise, you leak test data statistics into the transformation ("data leakage").

**Transform (.transform()):** Applies a previously learned transformation to new data. For example, `StandardScaler.transform()` applies the mean and standard deviation computed during `.fit()` to scale new data the same way.

Always use the same transformer object on training and test data to ensure consistent preprocessing.

**Standard Scaling (Z-score Normalization):** Transforms each feature to have **mean=0** and **standard deviation=1**:

$$z = \frac{x - \mu}{\sigma}$$

**Why scale?** Many ML algorithms (SVM, KNN, gradient descent, PCA) are sensitive to feature magnitudes. A feature ranging 0-1000 would dominate one ranging 0-1 without scaling. Tree-based models (Random Forest, XGBoost) are scale-invariant.

**Random Forest:** An ensemble method that builds many decision trees and averages their predictions (regression) or takes a majority vote (classification).

Two key tricks make it powerful:
1. **Bagging**: Each tree trains on a random subset of the data (with replacement)
2. **Feature randomness**: Each split considers only a random subset of features

This decorrelates the trees, reducing overfitting. Random Forests handle non-linear relationships, missing values, and mixed feature types well, with minimal tuning needed.

**Random Seed:** Setting a seed ensures **reproducibility** — the same random numbers are generated each time the code runs. This is critical in ML experiments because:
- Train/test splits will be the same
- Weight initialization will be identical
- Any randomized algorithm (dropout, data augmentation) will behave consistently

Without a fixed seed, your results would vary between runs, making it impossible to debug or compare experiments.

In [ ]:
# 2. Manual baseline (for comparison)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

print("Testing Manual RF baseline...")
start = time.time()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

manual_time = time.time() - start
manual_pred = rf.predict_proba(X_test_scaled)[:, 1]
manual_score = roc_auc_score(y_test, manual_pred)

results.append({
    'Platform': 'Manual RF',
    'Time (s)': manual_time,
    'ROC-AUC': manual_score,
    'Best Model': 'RandomForest'
})

print(f"✅ Manual RF: {manual_score:.4f} in {manual_time:.2f}s")

**Create a DataFrame:** Pandas DataFrames are 2D labeled data structures — they're the backbone of data science in Python. Each column can hold a different data type (numbers, strings, dates). DataFrames support powerful operations: filtering, grouping, merging, reshaping, and aggregation.

In [ ]:
# Compare results
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False)

print("\n" + "="*60)
print("AutoML Platform Comparison")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

Visualize comparison
ROC-AUC comparison
Time comparison

The code below implements this step in the low-code AI workflow.

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-AUC comparison
axes[0].barh(comparison_df['Platform'], comparison_df['ROC-AUC'], color='steelblue')
axes[0].set_xlabel('ROC-AUC Score')
axes[0].set_title('Performance Comparison')
axes[0].set_xlim(0.9, 1.0)

# Time comparison
axes[1].barh(comparison_df['Platform'], comparison_df['Time (s)'], color='coral')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Speed Comparison')

plt.tight_layout()
plt.show()

## Part 6: Best Practices and Tips

Run this cell and inspect the output to verify the low-code AI operations produce the expected results.

In [ ]:
print("""
AutoML Best Practices:

1. ✅ START WITH AUTOML for baselines
   - Get quick results
   - Understand data better
   - Identify promising models

2. ✅ SET REASONABLE TIME BUDGETS
   - Start with 60-300 seconds
   - Increase if needed
   - Balance speed vs performance

3. ✅ VALIDATE RESULTS
   - Check on holdout data
   - Look for overfitting
   - Understand model decisions

4. ✅ INSPECT TOP MODELS
   - Don't just use the best
   - Consider interpretability
   - Check robustness

5. ✅ USE FOR EXPLORATION
   - Try different features
   - Test hypotheses quickly
   - Compare preprocessing steps

6. ❌ DON'T BLINDLY TRUST
   - Review model choices
   - Understand limitations
   - Test edge cases

7. ❌ DON'T SKIP DATA CLEANING
   - AutoML isn't magic
   - Clean data = better results
   - Handle domain-specific issues

Platform Selection Guide:

Use PyCaret when:
  • Need comprehensive pipeline
  • Want visualization tools
  • Prefer low-code approach
  • Building prototypes

Use FLAML when:
  • Speed is critical
  • Limited compute resources
  • Want cost optimization
  • Production deployment

Use H2O when:
  • Very large datasets
  • Need distributed computing
  • Enterprise deployment
  • Java integration

Use Manual ML when:
  • Full control needed
  • Custom architectures
  • Specialized domains
  • Learning purposes
""")

## 🎯 Key Takeaways

1. **AutoML accelerates ML development** - Quick baselines and model exploration
2. **Multiple platforms available** - PyCaret (comprehensive), FLAML (fast), H2O (scalable)
3. **Not a silver bullet** - Still need data understanding and validation
4. **Great for baselines** - Perfect starting point before custom optimization
5. **Time-performance tradeoff** - More time generally = better models
6. **Interpretability matters** - Don't sacrifice understanding for slight accuracy gains

---

## 📝 Practice Exercises

1. **Compare AutoML Platforms**
   - Load a dataset
   - Run PyCaret, FLAML, and manual baseline
   - Compare results and insights

2. **Feature Engineering Impact**
   - Create new features
   - Run AutoML before/after
   - Measure improvement

3. **Time Budget Experiment**
   - Try different time budgets (10s, 60s, 300s)
   - Plot performance vs time
   - Find optimal budget

---

## 🔗 Resources

- [PyCaret Documentation](https://pycaret.org/)
- [FLAML Documentation](https://microsoft.github.io/FLAML/)
- [H2O AutoML Guide](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html)
- [AutoML Survey Paper](https://arxiv.org/abs/1810.13306)

---

**Next:** [Notebook 5 - End-to-End Low-Code Project](05_end_to_end_project.ipynb)